In [1]:
import sys
import os
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"
from IPython.display import display, HTML

# Set default Plotly template
pio.templates.default = "plotly_white"

import logging
# Stop showing "info" messages
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger('src.engine').setLevel(logging.WARNING)

# CSS to change report aesthetic
display(HTML("""
<style>
    body, .jp-Notebook, .notebook_app {
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif !important;
        background-color: #ffffff !important;
        color: #333333 !important;
    }
    .container, .jp-Notebook-cell { max-width: 950px !important; margin: 0 auto !important; }
    .jp-InputPrompt, .jp-OutputPrompt, div.prompt { display: none !important; width: 0 !important; }
    div.input { display: none !important; } /* Esconde o código Python */
    
    h1 { font-weight: 700; color: #111; margin-top: 40px !important; }
    h2 { font-weight: 600; color: #444; margin-top: 30px !important; border-bottom: 1px solid #eee; padding-bottom: 5px; }
    
    .kpi-card {
        background: #f8f9fa; border: 1px solid #e9ecef; border-radius: 8px;
        padding: 20px; margin: 10px 10px 10px 0; display: inline-block;
        min-width: 200px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);
    }
    .kpi-title { font-size: 12px; color: #6c757d; text-transform: uppercase; font-weight: 600; }
    .kpi-value { font-size: 28px; color: #212529; font-weight: 700; margin: 5px 0 0 0; }
    .kpi-sub { font-size: 12px; color: #adb5bd; }
</style>
"""))

# Add the 'src' directory to the path to import the custom modules
sys.path.append(os.path.abspath(os.path.join('..')))

from src.engine import RiskEngine
from src.data_gen import MarketDataGenerator, PortfolioGenerator

In [2]:
display(HTML("""
<div style="margin-top: 0px; margin-bottom: 0px;">
    <h1 style="color: #111; font-weight: 700; margin-bottom: 8px; border-bottom: 2px solid #3b82f6; padding-bottom: 5px;">
        Energy Risk Lab
    </h1>             

    <h2 style="color: #111; font-weight: 700; margin-bottom: 8px; border-bottom: 2px solid #3b82f6; display: inline-block; padding-bottom: 5px;">
        Project Setup & Data Generation
    </h2>
    
    <p style="color: #555; font-size: 16px; line-height: 1.5; margin-top: 0;">
        First, we initialize the Risk Engine and generate dummy market data (Forward Curves, Volatility) and a sample Trading Portfolio.
    </p>
</div>
"""))

In [3]:
from IPython.display import display, HTML

# --- 1. CONFIGURATION & GENERATION ---
BASE_DATE = pd.Timestamp('2025-01-01')
HORIZON_MONTHS = 24

fwd_curves = MarketDataGenerator.generate_forward_curves(BASE_DATE, HORIZON_MONTHS)
discount_curve = MarketDataGenerator.generate_discount_curve(BASE_DATE, HORIZON_MONTHS)
vertices = [f'M{i}' for i in range(1, 13)] + ['Y1']
risk_matrix = MarketDataGenerator.generate_risk_matrix(vertices)
portfolio_balance = PortfolioGenerator.generate_energy_balance(BASE_DATE, HORIZON_MONTHS)

# --- 2. DASHBOARD HEADER: PARAMETERS ---
display(HTML(f"""
<div style="display: flex; gap: 15px; margin-bottom: 0px;">
    <div class="kpi-card" style="border-left: 4px solid #3b82f6; flex: 1;">
        <div class="kpi-title">Base Date</div>
        <div class="kpi-value" style="font-size: 22px;">{BASE_DATE.strftime('%b %d, %Y')}</div>
        <div class="kpi-sub">Simulation Start</div>
    </div>
    <div class="kpi-card" style="border-left: 4px solid #3b82f6; flex: 1;">
        <div class="kpi-title">Forecast Horizon</div>
        <div class="kpi-value" style="font-size: 22px;">{HORIZON_MONTHS} Months</div>
        <div class="kpi-sub">Analysis Period</div>
    </div>
     <div class="kpi-card" style="border-left: 4px solid #3b82f6; flex: 1;">
        <div class="kpi-title">Risk Model</div>
        <div class="kpi-value" style="font-size: 22px;">Parametric</div>
        <div class="kpi-sub">Delta-Normal (Covariance)</div>
    </div>
</div>
"""))

# --- 3. MAIN CHART: MARKET SCENARIO ---

display(
    HTML("""
        <div style="margin-top: 0px; margin-bottom: 0px;">
        <h2
        style="color: #111; font-weight: 700; margin-bottom: 8px; border-bottom: 2px solid #3b82f6; display: inline-block; padding-bottom: 5px;">
        Energy Forward Curves (Simulated Scenario)
        </h2>
        </div>
"""))

fig_curve = px.line(
    fwd_curves.T, 
    labels={'index': 'Date', 'value': 'Price (BRL/MWh)', 'variable': 'Submarket'},
    color_discrete_sequence=px.colors.qualitative.G10
)
fig_curve.update_layout(
    xaxis_title="", yaxis_title="Price (BRL/MWh)",
    legend_title_text="", plot_bgcolor='white', hovermode="x unified",
    margin=dict(l=20, r=20, t=60, b=20), height=400
)
fig_curve.update_traces(line=dict(width=2.5))
fig_curve.show()

# --- 4. DATA SAMPLE ---
display(
    HTML("""
        <div style="margin-top: 0px; margin-bottom: 0px;">
        <h2
        style="color: #111; font-weight: 700; margin-bottom: 8px; border-bottom: 2px solid #3b82f6; display: inline-block; padding-bottom: 5px;">
        Strategy & Trading Portfolio (Sample)
        </h2>
         </div>
"""))

# Mostra apenas as primeiras 5 linhas para não poluir
display(portfolio_balance.head(5).style.format({'MW_Avg': '{:.2f}'}).hide(axis="index").set_properties(**{'background-color': '#f8f9fa', 'border': '1px solid #eee'}))

Date,Submarket,Source,Portfolio,MW_Avg
2025-01-01 00:00:00,NE,Solar,Trading,59.02
2025-01-01 00:00:00,SE,Contract,Trading,-36.13
2025-02-01 00:00:00,NE,Solar,Trading,59.09
2025-02-01 00:00:00,SE,Contract,Trading,-32.79
2025-03-01 00:00:00,NE,Solar,Trading,52.27


In [4]:
from IPython.display import display, HTML
display(HTML("""
<div style="margin-top: 0px; margin-bottom: 0px;">
    <h2 style="color: #111; font-weight: 700; margin-bottom: 8px; border-bottom: 2px solid #3b82f6; display: inline-block; padding-bottom: 5px;">
        Mark-to-Market (MtM) Calculation
    </h2>
    
    <p style="color: #555; font-size: 16px; line-height: 1.5; margin-top: 0;">
        We calculate the fair value of the portfolio by comparing the contracted positions against the current forward curves, discounting the cash flows to the Base Date.
    </p>
</div>
"""))

# Initialize Engine
engine = RiskEngine()

# --- Calculate Mark-to-Market (MtM) ---
# Calculates the liquid value of the portfolio against forward curves
mtm_matrix, total_mtm = engine.calculate_mtm(portfolio_balance, fwd_curves, discount_curve)

# --- VISUALIZATION: KPI CARD (Green Theme) ---
display(HTML(f"""
<div class="kpi-card" style="border-left: 4px solid #10b981;">
    <div class="kpi-title">Total Mark-to-Market (MtM)</div>
    <div class="kpi-value">R$ {total_mtm:,.2f}</div>
    <div class="kpi-sub">Net Present Value (NPV)</div>
</div>
"""))

# --- CHART: Monthly Financial Exposure ---
# Aggregating MtM by month to see cash flow exposure
mtm_per_month = mtm_matrix.sum()
df_mtm = mtm_per_month.to_frame(name='MtM Value')
df_mtm.index = df_mtm.index.strftime('%Y-%m')

# Define status for color coding (Green for Profit, Red for Loss)
df_mtm['Status'] = df_mtm['MtM Value'].apply(lambda x: 'Profit' if x >= 0 else 'Loss')

fig = px.bar(
    df_mtm, 
    x=df_mtm.index, 
    y='MtM Value', 
    color='Status',
    title='<b>Monthly Financial Exposure</b>',
    color_discrete_map={'Profit': '#10b981', 'Loss': '#ef4444'}, # Matte Green/Red
    text_auto='.2s'
)

# Clean Layout Configuration
fig.update_layout(
    xaxis_title="", 
    yaxis_title="BRL",
    showlegend=False,
    plot_bgcolor='white', # Clean white background
    hovermode="x unified",
    margin=dict(l=20, r=20, t=60, b=20)
)
fig.update_traces(textposition='outside', marker_line_width=0)
fig.show()

Parametric VaR Calculation

Here we estimate the Value at Risk (VaR) with 95% confidence. The engine maps the monthly exposures to the risk vertices (M1, M2... Y1) and applies the Delta-Normal approach.

In [5]:
# --- Calculate Parametric VaR ---

display(HTML("""
<div style="margin-top: 0px; margin-bottom: 0px;">
    <h2 style="color: #111; font-weight: 700; margin-bottom: 8px; border-bottom: 2px solid #3b82f6; display: inline-block; padding-bottom: 5px;">
        Parametric VaR Calculation
    </h2>
    
    <p style="color: #555; font-size: 16px; line-height: 1.5; margin-top: 0;">
        Here we estimate the Value at Risk (VaR) with 95% confidence. The engine maps the monthly exposures to the risk vertices (M1, M2... Y1) and applies the Delta-Normal approach.
    </p>
</div>
"""))

# Using the Delta-Normal approach (Exposure Vector * Covariance Matrix * Exposure Vector)
var_value, exposure_financial = engine.calculate_parametric_var(portfolio_balance, risk_matrix, BASE_DATE)

# --- VISUALIZATION: KPI CARD (Purple Theme for Risk) ---
display(HTML(f"""
<div class="kpi-card" style="border-left: 4px solid #8b5cf6;">
    <div class="kpi-title">Parametric VaR (95%)</div>
    <div class="kpi-value">R$ {var_value:,.2f}</div>
    <div class="kpi-sub">Delta-Normal Approach (1-Day Horizon)</div>
</div>
"""))

# --- CHART 1: Exposure Vector by Vertex ---
fig_exp = px.bar(
    exposure_financial, 
    orientation='h', # Horizontal bars read better for labels
    title='<b>Net Financial Exposure Vector (s)</b>',
    labels={'index': 'Risk Vertex', 'value': 'Exposure (BRL)'},
    color_discrete_sequence=['#8b5cf6'] # Modern Purple
)

fig_exp.update_layout(
    showlegend=False, 
    plot_bgcolor='white',
    margin=dict(l=20, r=20, t=60, b=20)
)
fig_exp.update_traces(marker_line_width=0)
fig_exp.show()

# --- CHART 2: Covariance Matrix Heatmap ---
fig_heat = px.imshow(
    risk_matrix, 
    text_auto='.2f', 
    aspect="auto",
    title='<b>Covariance Matrix (Volatility & Correlation)</b>',
    color_continuous_scale='RdBu_r', # Monochromatic Blue (Cleaner look)
    origin='lower'
)

fig_heat.update_xaxes(side="bottom")
fig_heat.update_layout(
    coloraxis_showscale=False, # Hides the scale bar for a cleaner UI
    margin=dict(l=20, r=20, t=60, b=20)
)
fig_heat.show()